# Hub Results Mapping with Folium

This notebook visualizes the final hub prioritization results on an interactive map using Folium.

- **Polygons**: Blue (hub areas)
- **Centroids**: Black markers (hub centers)
- **Tooltips**: Display group ID on hover

## 1. Import Libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium import Tooltip
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load Data

In [ ]:
# Load the CSV with hub results
# Update this path to point to your final results CSV
csv_path = 'grouped_hubs_ready_for_scoring_21082025.csv'

df = pd.read_csv(csv_path, encoding='utf-8')

print(f"Loaded {len(df)} hubs from {csv_path}")
print(f"\nColumns: {list(df.columns)}")
df.head(3)

## 3. Convert to GeoDataFrame

In [ ]:
# Convert WKT geometry strings to shapely geometries
df['geometry'] = df['geometry'].apply(wkt.loads)
df['centroid_geom'] = df['centroid'].apply(wkt.loads)

# Create GeoDataFrame for polygons
gdf_polygons = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Create GeoDataFrame for centroids
gdf_centroids = gpd.GeoDataFrame(df, geometry='centroid_geom', crs='EPSG:4326')

print(f"Created GeoDataFrames:")
print(f"  - Polygons: {len(gdf_polygons)} features")
print(f"  - Centroids: {len(gdf_centroids)} features")
print(f"\nCRS: {gdf_polygons.crs}")

## 4. Calculate Map Center

In [ ]:
# Calculate the center of all hubs for initial map view
center_lat = gdf_centroids.geometry.y.mean()
center_lon = gdf_centroids.geometry.x.mean()

print(f"Map center: ({center_lat:.4f}, {center_lon:.4f})")

## 5. Create Folium Map

In [ ]:
# Create base map centered on Israel
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8,
    tiles='OpenStreetMap'
)

# Add alternative tile layers
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Dark').add_to(m)

print("Base map created successfully!")

## 6. Add Hub Polygons (Blue)

In [ ]:
# Add polygons to map
for idx, row in gdf_polygons.iterrows():
    # Create tooltip with group ID
    tooltip_text = f"Group ID: {row['group']}"
    
    # Add additional info if available
    if 'HubType' in row and pd.notna(row['HubType']):
        tooltip_text += f"<br>Type: {row['HubType']}"
    if 'TotalDemand' in row and pd.notna(row['TotalDemand']):
        tooltip_text += f"<br>Demand: {row['TotalDemand']:.0f}"
    
    # Create polygon
    folium.GeoJson(
        row['geometry'],
        style_function=lambda x: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.3
        },
        tooltip=folium.Tooltip(tooltip_text)
    ).add_to(m)

print(f"Added {len(gdf_polygons)} polygon features to map")

## 7. Add Hub Centroids (Black)

In [ ]:
# Add centroids as markers
for idx, row in gdf_centroids.iterrows():
    # Create tooltip with group ID
    tooltip_text = f"Group ID: {row['group']}"
    
    # Add additional info if available
    if 'HubType' in row and pd.notna(row['HubType']):
        tooltip_text += f"<br>Type: {row['HubType']}"
    if 'address' in row and pd.notna(row['address']):
        tooltip_text += f"<br>Location: {row['address'][:50]}..."
    if 'TotalDemand' in row and pd.notna(row['TotalDemand']):
        tooltip_text += f"<br>Demand: {row['TotalDemand']:.0f}"
    if 'Num_Modes' in row and pd.notna(row['Num_Modes']):
        tooltip_text += f"<br>Modes: {int(row['Num_Modes'])}"
    
    # Add circle marker for centroid
    folium.CircleMarker(
        location=[row['centroid_geom'].y, row['centroid_geom'].x],
        radius=5,
        color='black',
        fill=True,
        fillColor='black',
        fillOpacity=0.8,
        weight=2,
        tooltip=folium.Tooltip(tooltip_text)
    ).add_to(m)

print(f"Added {len(gdf_centroids)} centroid markers to map")

## 8. Add Layer Control and Display Map

In [ ]:
# Add layer control
folium.LayerControl().add_to(m)

# Add title
title_html = '''
             <div style="position: fixed; 
             top: 10px; left: 50px; width: 400px; height: 60px; 
             background-color: white; border:2px solid grey; z-index:9999; 
             font-size:16px; padding: 10px">
             <b>Hub Prioritization Results</b><br>
             <span style="color:blue">■</span> Hub Polygons &nbsp;&nbsp;
             <span style="color:black">●</span> Hub Centroids
             </div>
             '''
m.get_root().html.add_child(folium.Element(title_html))

print("Map ready to display!")
m

## 9. Save Map to HTML

In [ ]:
# Save map to HTML file
output_path = 'hub_results_map.html'
m.save(output_path)

print(f"Map saved to: {output_path}")
print(f"You can open this file in any web browser to view the interactive map.")

## 10. Summary Statistics

In [ ]:
# Display summary statistics
print("=" * 60)
print("HUB MAPPING SUMMARY")
print("=" * 60)
print(f"Total hubs mapped: {len(df)}")

if 'HubType' in df.columns:
    print("\nHub Type Distribution:")
    print(df['HubType'].value_counts())

if 'area' in df.columns:
    print("\nRegional Distribution:")
    print(df['area'].value_counts())

if 'TotalDemand' in df.columns:
    print("\nDemand Statistics:")
    print(f"  Mean: {df['TotalDemand'].mean():.0f}")
    print(f"  Median: {df['TotalDemand'].median():.0f}")
    print(f"  Min: {df['TotalDemand'].min():.0f}")
    print(f"  Max: {df['TotalDemand'].max():.0f}")

print("=" * 60)